In [1]:
import collections, json, math, os, re, time

import anthropic
import grpc
from IPython.display import Markdown, display
import networkx as nx
import openai
import pandas as pd
from senzing import SzEngineFlags as F
from senzing_grpc import SzAbstractFactoryGrpc
from sqlalchemy import create_engine

HYATT = 2120          # the conference hotel - every question starts here

In [2]:
channel = grpc.insecure_channel(
    f"{os.getenv('SENZING_GRPC_HOST', 'resolver')}:8261",
    options=[('grpc.max_receive_message_length', 200 * 1024 * 1024)])
grpc.channel_ready_future(channel).result(timeout=20)
engine = SzAbstractFactoryGrpc(channel).create_engine()
print("✅ resolver connected")

✅ resolver connected


In [3]:
db = create_engine(
    f"postgresql+psycopg2://{os.getenv('POSTGRES_USER', 'postgres')}:"
    f"{os.getenv('POSTGRES_PASSWORD', 'workshop')}@"
    f"{os.getenv('POSTGRES_HOST', 'postgres')}:5432/{os.getenv('POSTGRES_DB', 'erkg')}")
print("✅ database connected")

✅ database connected


In [4]:
geo = pd.read_sql("""
    SELECT entity_id, street, city, placekey, latitude, longitude
    FROM workshop.entity_geo
    WHERE placekey IS NOT NULL AND street NOT LIKE 'PO BOX%%'
""", db)

# All addresses per entity, not just one. Where they disagree is interesting.
addresses = collections.defaultdict(list)
for r in geo.itertuples():
    addresses[int(r.entity_id)].append(
        {'street': r.street, 'city': r.city, 'placekey': r.placekey,
         'lat': r.latitude, 'lon': r.longitude})

print(f"{len(addresses):,} entities have a placekey")

2,376 entities have a placekey


In [5]:
t0 = time.time()
G = nx.Graph()
pending_links = []

handle = engine.export_json_entity_report(F.SZ_EXPORT_DEFAULT_FLAGS)
try:
    while (line := engine.fetch_next(handle)):
        d = json.loads(line)
        e = d['RESOLVED_ENTITY']
        eid = e['ENTITY_ID']
        recs = e.get('RECORDS', [])
        a = addresses.get(eid, [])

        G.add_node(eid,
                   name=e.get('ENTITY_NAME') or f'Entity {eid}',
                   sources=sorted({r['DATA_SOURCE'] for r in recs}),
                   n_records=len(recs),
                   records=[f"{r['DATA_SOURCE']}:{r['RECORD_ID']}" for r in recs[:8]],
                   addresses=a,
                   placekey=a[0]['placekey'] if a else None,
                   street=a[0]['street'] if a else None,
                   city=a[0]['city'] if a else None,
                   lat=a[0]['lat'] if a else None,
                   lon=a[0]['lon'] if a else None)

        for r in d.get('RELATED_ENTITIES', []):
            pending_links.append((eid, r['ENTITY_ID'], r.get('MATCH_KEY', ''),
                                  bool(r.get('IS_AMBIGUOUS', 0))))
finally:
    engine.close_export_report(handle)

print(f"{G.number_of_nodes():,} entity nodes in {time.time() - t0:.0f}s")

31,339 entity nodes in 21s


In [6]:
for a, b, match_key, ambiguous in pending_links:
    if a < b and G.has_node(b):          # a < b so each pair is added once
        G.add_edge(a, b, match_key=match_key, ambiguous=ambiguous)

print(f"{G.number_of_edges():,} entity-resolution edges")

11,822 entity-resolution edges


In [7]:
located = [n for n, a in G.nodes(data=True) if a['lat'] is not None]
keys = collections.Counter(d['match_key'] for _, _, d in G.edges(data=True))

print(f"entities          {G.number_of_nodes():,}")
print(f"  with a placekey {len(located):,}")
print(f"ER edges          {G.number_of_edges():,}")
print(f"  ambiguous       {sum(1 for _,_,d in G.edges(data=True) if d['ambiguous']):,}")
print(f"\nmatch keys: {keys.most_common(6)}")
print(f"\nanchor: {G.nodes[HYATT]['name']} at {G.nodes[HYATT]['street']}")

entities          31,339
  with a placekey 2,376
ER edges          11,822
  ambiguous       24

match keys: [('+ADDRESS-RECORD_TYPE', 5763), ('+ADDRESS', 2396), ('+NAME', 2270), ('+ADDRESS+REGISTRATION_DATE', 704), ('+SURNAME+ADDRESS', 212), ('+ADDRESS+PHONE', 157)]

anchor: HYATT REGENCY SAN FRANCISCO AIRPORT at 1333 BAYSHORE HWY


In [8]:
def cell(placekey):
    """A placekey is what@where. The 'where' is a ~76 m H3 hexagon, so entities
    sharing it are in the same building."""
    return placekey.split('@')[-1]


def metres(a, b):
    x, y = G.nodes[a], G.nodes[b]
    la1, lo1, la2, lo2 = x['lat'], x['lon'], y['lat'], y['lon']
    return 6371000 * 2 * math.asin(math.sqrt(
        math.sin(math.radians(la2 - la1) / 2) ** 2 +
        math.cos(math.radians(la1)) * math.cos(math.radians(la2)) *
        math.sin(math.radians(lo2 - lo1) / 2) ** 2))


print(f"{metres(HYATT, 1259):.0f} m to entity 1259 (same building)")

0 m to entity 1259 (same building)


In [9]:
def neighbourhood(anchor=HYATT, radius_m=600, max_places=25):
    """Every located entity within radius_m of the anchor, grouped by building."""
    anchor_cell = cell(G.nodes[anchor]['placekey'])
    near = sorted(((n, metres(anchor, n)) for n in located if n != anchor),
                  key=lambda t: t[1])

    by_place = collections.OrderedDict()
    for n, d in near:
        c = cell(G.nodes[n]['placekey'])
        by_place.setdefault(c, {'d': d, 'ents': []})
        if d <= radius_m or c == anchor_cell:
            by_place[c]['ents'].append(n)

    places = {c: v for c, v in by_place.items() if v['ents']}
    places = dict(list(places.items())[:max_places])
    members = {anchor} | {n for v in places.values() for n in v['ents']}
    return places, members


places, members = neighbourhood()
print(f"{len(places)} locations, {len(members)} entities")

25 locations, 171 entities


In [10]:
def describe(anchor=HYATT, radius_m=600, max_places=200):
    places, members = neighbourhood(anchor, radius_m, max_places)
    A = G.nodes[anchor]

    out = [f"STARTING POINT: {A['name']} (entity {anchor})",
           f"  {A['street']}, {A['city']} | placekey {A['placekey']} "
           f"| sources: {', '.join(A['sources'])}",
           f"  source records merged into this one entity: {', '.join(A['records'])}",
           "",
           "SURROUNDING LOCATIONS - metres from the starting point, computed from",
           "Placekey coordinates. Businesses under one heading share a Placekey",
           "cell, i.e. they are in the same building.", ""]

    for c, v in places.items():
        streets = sorted({a['street'] for n in v['ents'] for a in G.nodes[n]['addresses']
                          if cell(a['placekey']) == c})
        out.append(f"{v['d']:.0f} m - {streets[0]}  [cell {c}]")
        for n in v['ents'][:12]:
            a = G.nodes[n]
            out.append(f"    {n} {a['name']} | {', '.join(a['sources'])} "
                       f"| {a['n_records']} record(s)")
        if len(v['ents']) > 12:
            out.append(f"    ... and {len(v['ents']) - 12} more")

    out += ["", "ENTITY RESOLUTION LINKS. Senzing decided these are related; the key",
            "says why. +ADDRESS alone means only a shared address. +NAME or +PHONE",
            "means the match would survive the business moving premises."]

    inner, outer = [], []
    for m in sorted(members):
        for nb in G.neighbors(m):
            if nb < m and nb in members:
                continue
            (inner if nb in members else outer).append((m, nb, G.edges[m, nb]))

    # Most links are +ADDRESS, which is near-tautological with sharing a placekey.
    # Sort the address-independent ones first so truncation cannot discard them.
    def interest(t):
        d = t[2]
        return (0 if d['ambiguous'] else 1,
                0 if ('+NAME' in d['match_key'] or '+PHONE' in d['match_key']) else 1)

    inner.sort(key=interest)
    outer.sort(key=interest)

    def fmt(m, nb, d):
        return (f"  {m} {G.nodes[m]['name'][:32]} <-> {nb} {G.nodes[nb]['name'][:32]}"
                f"  {d['match_key']}{' (AMBIGUOUS)' if d['ambiguous'] else ''}")

    out.append("  -- between businesses in this neighbourhood:")
    out += [fmt(m, nb, d) for m, nb, d in inner[:50]]

    out.append("  -- reaching OUTSIDE this neighbourhood:")
    for m, nb, d in outer[:30]:
        b = G.nodes[nb]
        if b['lat'] is not None:
            out.append(fmt(m, nb, d) +
                       f"  [{b['street']}, {metres(m, nb):.0f} m from here]")
        else:
            out.append(fmt(m, nb, d) + "  [no address on file]")

    out += ["", "DATA QUALITY. A Placekey is what@where. A long 'what' beginning with 0",
            "means the address could not be matched, so its coordinates are unreliable."]
    for n in sorted(members):
        a = G.nodes[n]
        cells = {cell(x['placekey']) for x in a['addresses']}
        if len(cells) > 1:
            out.append(f"  entity {n} {a['name']} has {len(a['addresses'])} addresses "
                       f"in {len(cells)} different cells:")
            out += [f"      {x['street']} -> {x['placekey']}" for x in a['addresses']]

    return "\n".join(out)

In [11]:
ctx = describe()
print(f"{len(ctx):,} characters (~{len(ctx)//4:,} tokens)\n")
print(ctx[:1400])

23,351 characters (~5,837 tokens)

STARTING POINT: HYATT REGENCY SAN FRANCISCO AIRPORT (entity 2120)
  1333 BAYSHORE HWY, BURLINGAME | placekey 222@5vg-7y4-wx5 | sources: ABC-RETAIL, BURLINGAME
  source records merged into this one entity: BURLINGAME:20500020, ABC-RETAIL:220571-47, ABC-RETAIL:220571-58

SURROUNDING LOCATIONS - metres from the starting point, computed from
Placekey coordinates. Businesses under one heading share a Placekey
cell, i.e. they are in the same building.

0 m - 1333 BAYSHORE HWY  [cell 5vg-7y4-wx5]
    1259 CURTIN CONVENTION & EXPOSITION SERVICE | BURLINGAME | 1 record(s)
    1561 ENCORE GROUP USA LLC | BURLINGAME | 1 record(s)
    1993 HALLMARK MARKETING CORPORATION | BURLINGAME | 1 record(s)
    2202 IRON MOUNTAIN | BURLINGAME | 1 record(s)
    4394 TRAVEL TRADERS #698 | BURLINGAME | 1 record(s)
131 m - 851 BURLWAY 519  [cell 5vg-7y4-wff]
    497 B Y & S CPAS A PROFESSIONAL CORPORATION | BURLINGAME | 1 record(s)
    564 BAY AREA SERVICE | BURLINGAME | 1 reco

In [12]:
anthropic_client = anthropic.Anthropic() if os.getenv("ANTHROPIC_API_KEY") else None
openai_client    = openai.OpenAI()       if os.getenv("OPENAI_API_KEY")    else None

ANTHROPIC_MODEL = "claude-opus-5"
OPENAI_MODEL    = "gpt-5.4-nano"

print("Anthropic:", anthropic_client is not None, " OpenAI:", openai_client is not None)

Anthropic: True  OpenAI: True


In [13]:
PLANNER_MODEL = "claude-haiku-4-5-20251001"

PLANNER_PROMPT = (
    "You read a search radius, in metres, out of a question about businesses "
    "near a hotel. Reply with ONLY an integer, or ONLY the word NONE.\n\n"
    "If the question states a distance, time or count, convert it exactly:\n"
    "  '1000m' -> 1000       '300 m' -> 300        '2 km' -> 2000\n"
    "  'half a mile' -> 805  'a mile' -> 1609      '3 blocks' -> 360\n"
    "  '10 minute walk' -> 800\n"
    "  (a block is 120 m; walking is 80 m per minute)\n\n"
    "If it states no number but describes a distance, use:\n"
    "  'the same building', 'the same address' -> 1\n"
    "  'next door', 'two doors down', 'immediately outside' -> 250\n"
    "  'near', 'nearby', 'close by', 'around here' -> 600\n"
    "  'walk', 'walking distance', 'on foot' -> 800\n"
    "  'this neighbourhood', 'the area', 'the wider area' -> 1500\n"
    "  'anywhere', 'in Burlingame', 'the whole town' -> 3000\n\n"
    "Any mention of a building, address, premises, walking or proximity IS a "
    "distance - map it above, never NONE.\n"
    "Reply NONE only when the question refers to no place or distance at all - "
    "for example about data quality, which records merged, or how businesses "
    "are connected to each other.\n\n"
    "Worked examples:\n"
    "  Q: What else operates out of the same building as the hotel?  A: 1\n"
    "  Q: Who shares this address?  A: 1\n"
    "  Q: Anywhere I could walk to for a drink?  A: 800\n"
    "  Q: What are my options within 1000m?  A: 1000\n"
    "  Q: Which links is the system unsure about?  A: NONE\n"
    "  Q: What records merged into the hotel entity?  A: NONE\n\n"
    "Reply with a single integer between 1 and 5000, or NONE. Nothing else."
)


def radius_for(question):
    """Read the distance out of the question. Returns None when the question
    contains no distance at all - there is no fallback value."""
    if anthropic_client:
        r = anthropic_client.messages.create(
            model=PLANNER_MODEL, max_tokens=2000,
            system=PLANNER_PROMPT,
            messages=[{"role": "user", "content": question}])
        text = "".join(b.text for b in r.content if b.type == "text").strip()
    else:
        r = openai_client.chat.completions.create(
            model=OPENAI_MODEL, max_completion_tokens=2000,
            messages=[{"role": "system", "content": PLANNER_PROMPT},
                      {"role": "user", "content": question}])
        text = (r.choices[0].message.content or "").strip()

    if re.search(r'\bNONE\b', text, re.I):
        return None
    m = re.search(r'\d+', text)
    return max(1, min(int(m.group()), 5000)) if m else None


for q in ["What are my options within 1000m?",
          "What else operates out of the same building as the hotel?",
          "Anywhere I could walk to for a drink?",
          "Which links is the system unsure about?"]:
    print(f"  {str(radius_for(q)):>5}   {q}")


   1000   What are my options within 1000m?
      1   What else operates out of the same building as the hotel?
    800   Anywhere I could walk to for a drink?
   None   Which links is the system unsure about?


In [19]:
SYSTEM_PROMPT = """You answer questions about businesses around a hotel, using a knowledge 
    graph built from California public records. The records were entity-resolved 
    by Senzing - one entity is one real business, built from several source 
    records - and geocoded with Placekey. 
    
    DISTANCES ARE EXACT: straight-line metres computed from Placekey 
    coordinates, not guessed from street names. Quote them. Treat 400 m as a 
    short walk, 800 m as about ten minutes. 
    
    Businesses sharing a Placekey cell are in the same building.
    
    ABC-RETAIL and ABC-NONRETAIL mean a liquor licence; no other source implies 
    alcohol. 
    
    Links marked +ADDRESS, +NAME or +PHONE were made by entity resolution - say 
    which key supports a claim. AMBIGUOUS means Senzing is unsure. 
    Answer only from the context, cite entity IDs, and say plainly when the 
    context does not support an answer.
"""

In [20]:
def ask(question, provider="anthropic", max_tokens=8000):
    """Ask anything about the hotel's surroundings. The distance comes from the
    question; if the question has none, the search area is the whole dataset."""
    radius_m = radius_for(question)

    if radius_m is None:
        # No distance in the question, so do not invent one - search everything
        # that has coordinates and say so.
        radius_m = 5000
        scope = "no distance given, so searching every geocoded business"
    else:
        scope = "from the question"

    context = describe(radius_m=radius_m)

    print(f"Question: {question}")
    print(f"Searching within {radius_m} m ({scope})")
    print(f"Context: {len(context):,} chars (~{len(context)//4:,} tokens)")

    msg = f"Knowledge graph context:\n{context}\n\nQuestion: {question}"

    if provider == "anthropic":
        r = anthropic_client.messages.create(
            model=ANTHROPIC_MODEL, max_tokens=max_tokens,
            system=SYSTEM_PROMPT, messages=[{"role": "user", "content": msg}])
        answer = "".join(b.text for b in r.content if b.type == "text")
        if not answer:
            print(f"⚠️  no text returned. stop_reason={r.stop_reason}, "
                  f"blocks={[b.type for b in r.content]}, "
                  f"output_tokens={r.usage.output_tokens}")
            return
    else:
        r = openai_client.chat.completions.create(
            model=OPENAI_MODEL, max_completion_tokens=max_tokens,
            messages=[{"role": "system", "content": SYSTEM_PROMPT},
                      {"role": "user", "content": msg}])
        answer = r.choices[0].message.content
        if not answer:
            print(f"⚠️  no text returned. finish_reason={r.choices[0].finish_reason}")
            return

    # The model answers in markdown, so render it rather than printing it raw.
    display(Markdown(answer))


In [21]:
ask("I'm staying at this hotel. What else operates out of the same building?")

Question: I'm staying at this hotel. What else operates out of the same building?
Searching within 1 m (from the question)
Context: 2,473 chars (~618 tokens)


**Same building as the Hyatt Regency San Francisco Airport (entity 2120), 1333 Bayshore Hwy, Burlingame — Placekey cell `222@5vg-7y4-wx5`, 0 m away:**

| Entity | Business | Source | Likely role |
|---|---|---|---|
| 1259 | CURTIN CONVENTION & EXPOSITION SERVICE | BURLINGAME (1 record) | Trade-show / convention services |
| 1561 | ENCORE GROUP USA LLC | BURLINGAME (1 record) | Event AV / production services |
| 1993 | HALLMARK MARKETING CORPORATION | BURLINGAME (1 record) | Retail / cards & gifts |
| 2202 | IRON MOUNTAIN | BURLINGAME (1 record) | Records storage / shredding |
| 4394 | TRAVEL TRADERS #698 | BURLINGAME (1 record) | Hotel gift shop operator |

All five are tied to the hotel entity only by **+ADDRESS** links — Senzing matched them on the shared 1333 Bayshore Hwy address, not on name or phone. So the graph supports "they operate at the hotel's address/building," but it doesn't independently confirm what each one does on site; the descriptions above are inferred from the business names, not from the records.

Two things worth noting:

- The mix (convention services, event AV, a gift-shop operator) is what you'd expect of in-house concessions and vendors at a large convention hotel. Iron Mountain is more likely a service account registered to the address than a storefront you can visit.
- **ENCORE GROUP USA LLC (1561)** also links outward to **entity 600708**, the same company elsewhere, via **+NAME** — a match that would survive the business moving premises. That outside record has no address on file.

On alcohol: only the Hyatt itself (entity 2120) carries ABC-RETAIL licences (records 220571-47 and 220571-58). None of the five co-tenants has a liquor licence in this data, so the hotel's own bar/restaurant operation is the licensed one in the building.

In [22]:
ask("Where's the nearest place I could get a drink, and how far is the walk?")

Question: Where's the nearest place I could get a drink, and how far is the walk?
Searching within 800 m (from the question)
Context: 30,833 chars (~7,708 tokens)


**You don't have to leave the building.**

**Entity 2120 — HYATT REGENCY SAN FRANCISCO AIRPORT itself, 1333 Bayshore Hwy, holds an ABC-RETAIL licence.** Two of the three source records merged into that entity are liquor-licence records (ABC-RETAIL:220571-47 and ABC-RETAIL:220571-58), alongside the Burlingame business record. So the hotel bar is 0 m away.

If you want to go out instead, nearest licensed neighbours:

| Entity | Name | Distance | Evidence |
|---|---|---|---|
| 2783 | MAX'S OF BURLINGAME, 1250 Bayshore Hwy | **263 m** (~3 min walk) | ABC-RETAIL + BURLINGAME, 3 merged records; shares cell 5vg-7y4-nkf with Holiday Inn Express SFO South (2090) |
| 1918 | GRAND HARBOR SEAFOOD AND DIMSUM | 409 m *(uncertain — see below)* | ABC-RETAIL + BURLINGAME |
| 653 | BENIHANA OF TOKYO INC, 1496 Bayshore Hwy | 486 m *(uncertain)* | ABC-RETAIL + BURLINGAME |
| 1975 | H L PENINSULA PEARL, 1515 Bayshore Hwy | 577 m | ABC-RETAIL + BURLINGAME |

Caveats on the two "uncertain" ones: entity 1918 has two address records in different Placekey cells, and the one giving 409 m has a long unmatched 'what' prefix (04ynd7uxcx@…), so its coordinates are unreliable — its other record places it at cell 5vg-7y4-t9z, i.e. about 355 m. Entity 653 Benihana likewise has a second address (1494 Bayshore Hwy) with an unmatched Placekey, so treat 486 m as approximate.

Bottom line: a drink at the Hyatt is 0 m; Max's of Burlingame at **263 m** is the nearest licensed place off-property.

In [23]:
ask("Which businesses around the hotel are connected to each other, "
    "and what evidence connects them?")

Question: Which businesses around the hotel are connected to each other, and what evidence connects them?
Searching within 5000 m (no distance given, so searching every geocoded business)
Context: 62,690 chars (~15,672 tokens)


## Connections in this neighbourhood, by strength of evidence

Senzing's link key tells you how much weight to give each connection. **+NAME or +PHONE** links would survive the business moving; **+ADDRESS alone** only says "same premises"; **AMBIGUOUS** means Senzing itself is unsure.

---

### 1. Same-business-in-two-source-files links (+NAME) — the strongest

| Link | Where | Distance |
|---|---|---|
| **1254 CUISINE STYLE** ↔ **100061 CUISINE STYLE** (ABC-RETAIL) | both 810 Stanton Rd, cell 5vg-7y4-tjv | 862 m |
| **3852 SEGAN INDUSTRIES INC** ↔ **3853 SEGAN INDUSTRIES INC** | both 810 Stanton Rd, same cell | 862 m |
| **1391 DIVINEY CONSTRUCTION INC** ↔ **603131 DIVINEY CONSTRUCTION INC** (CA-CONTRACTORS) | both 802 Burlway Rd, cell 5vg-7y4-whq | 263 m |
| **2832 / 2837 META PLATFORMS INC** | both 1832 Rollins Rd | 1,500 m |
| **3212 / 3213 PACKARD CHILDRENS HEALTH ALLIANCE** | 1720 El Camino Real | 1,455 m |
| **3225, 3227, 3228, 3229, 3230, 3231 PALO ALTO MEDICAL FOUNDATION** — a six-entity all-to-all +NAME cluster | 1720 El Camino Real, same cell as Packard | 1,455 m |
| **308 ↔ 309 AMERICAN MEDICAL RESPONSE WEST** | **different buildings**: 309 at 1510 Rollins Rd (485 m), 308 at 1616 Rollins Rd (920 m) | 485 m / 920 m |
| **427 ART OF ITALY INC** ↔ **602815 ART OF ITALY INC** | 1003 Edwards Rd | 495 m |
| **2240 JAB CONSTRUCTION CO LTD** ↔ **601959 J A B CONSTRUCTION RESTORATION &** | — | — |

The AMR pair (308/309) is the interesting one: the name match holds across two separate Rollins Rd buildings 435 m apart, so it looks like one operator on two sites rather than a duplicate record.

Note for the bar list: the Cuisine Style +NAME link is what ties the plain city-registration record (1254) to the **ABC-RETAIL licence** record (100061) — that is the only source here that implies alcohol.

### 2. Shared address *and* phone (+ADDRESS+PHONE) — different names, almost certainly the same operation

- **1825 GENERATION BUILDERS INC** ↔ **601370 GB PLUMBING & CONSTRUCTION LLC** — 1003 Edwards Rd, **495 m**. Same phone, so not just a shared mailbox.
- **2723 MARCHETTI GROUP** ↔ **603225 MORRONE CONSTRUCTION** — 890 Cowan Rd, **1,333 m**.
- **600206 OTOOLE CONSTRUCTION INC** ↔ **602668 D O T CONSTRUCTION** ↔ **603089 DOT CONCRETE** — a three-way ring, all at 1657 Balboa Way, **1,564 m**. Three contractor licences, one address, one phone.

### 3. Salon clusters — name+address but flagged AMBIGUOUS

- **1044 CHIC** ↔ **1045 CHIC SALON – LUCY SCHUSSOLIN** ↔ **1046 CHIC SALON – NAILS BY LINDA** (AMBIGUOUS). The context does not give me an address or distance for these three.
- **2559 LE REVE SALON** ↔ **2560 HAIR BY JUDY**, **2561 HOURIG KIZIRIAN**, **2562 JESSI SEISA** — all at 809 California Dr, **1,335 m**. Classic chair-renter pattern: individual stylists registered at the salon.

### 4. Shared address only (+ADDRESS) — co-tenancy, nothing more

- **1200 Capuchino Ave, 1,025 m**: **4 / 5 (1202 HAIR STUDIO)**, **1604 ESSENTIAL HARI SALON**, **4154 SUZETTE HANNA** all interlinked by address alone.
- **1501 Adrian Rd, 354 m**: **8 "1501 ADRIAN RD LLC"** ↔ **4269 THE FLOOR STORE** — reads as landlord entity plus tenant.
- **1799 Bayshore Hwy #230, 1,122 m**: a large address-only web around **9 "1799 BUILDING – GELLER & PREINER"**, linking **33 7PIERS CAPITAL**, **322 AMIN DEAN CPA**, **464 AUDIT AND TAX SOLUTIONS CPAS**, **1820 GELLER INTERNATIONAL**, **3015 NETLINX TRANSPORT**, **3448 PROICON TRANSPORTATION**, **3929 SIEGWART GERMAN AMERICAN LAW**, **4709 WORLDNET CHARTER**. This is a suite-share/registered-agent address; I would not claim any business relationship between them from this evidence alone.

### 5. Links reaching outside the mapped area (+NAME)

Several neighbours have a twin record elsewhere in the state file:

- **1561 ENCORE GROUP USA LLC** — inside the hotel itself, **0 m**, cell 5vg-7y4-wx5 — ↔ **600708 ENCORE GROUP USA LLC** (no address on file).
- **2669 LUXOR FLOORS INC** (1101 Broadway, 813 m) **and** **2670 LUXOR FLOORS INC** (1717 Adrian Rd, 1,171 m) both ↔ **600084 LUXOR FLOORS INC** at 1325 Howard Ave — one firm, two local shopfronts, one licence record.
- **36 836-868 COWAN RD LLC** (868 Cowan Rd, 1,383 m) ↔ **35** same name at 836 Cowan Rd, 3,328 m.
- **2484 KS CHANNING ROAD ELECTRIC** (1291 Whitehorn Way, 486 m) ↔ **601891** at 18 Channing Rd, 2,874 m.
- Also: **81 AA CUSTOMS PROAV**, **124 ACE PARKING**, **564 BAY AREA SERVICE**, **773 BRIGHTVIEW LANDSCAPE**, **895 CAL STEAM**, **901 CALICO PLUMBING**, **1134 COIT SERVICES → C & S DRAPERIES**, **1331 DELAHOUSSAYE**, **1510 EK ROBIN CONSTRUCTION → EK CONSTRUCTION**, **1694 FIREMASTER**, **2094 HOME**, **2404 KDDI AMERICA**, **3163 ORKIN**, **3272 PAUL COFFEY CONSTRUCTION**, **3934 SIGNATURE CONSTRUCTION**, **2869 MID PENINSULA ROOFING → PENINSULA ROOFING**, **3011 NESCO**, **3640 RIVER ENGINEERING**, **2888 MILLER DEVELOPMENT**, **2382 KAMRAN EHSANIPOUR AIA** — all +NAME to records with a PO-box or no address.

### Caveats

- Nothing in this context links any of these businesses to the **Hyatt Regency (entity 2120)** itself beyond the five firms sharing its building (1259 Curtin Convention & Exposition Service, 1561 Encore Group, 1993 Hallmark Marketing, 2202 Iron Mountain, 4394 Travel Traders #698 — all 0 m, cell 5vg-7y4-wx5). Senzing drew no resolution link between the hotel and them; the only evidence is the shared Placekey cell.
- Treat the second address of **653 Benihana**, **1359 Devincenzi**, **1918 Grand Harbor Seafood**, **2076 Hills Pool Service** and **3182 P L Annuzzi** with suspicion — their alternate Placekeys start with a long unmatched "what" string, so those coordinates are unreliable; use the 222-prefixed cell.

In [25]:
ask("Are any of the businesses near the hotel part of something larger "
    "elsewhere in California?")

Question: Are any of the businesses near the hotel part of something larger elsewhere in California?
Searching within 5000 m (no distance given, so searching every geocoded business)
Context: 62,690 chars (~15,672 tokens)


## Yes — several, and the knowledge graph shows it in three different ways

### 1. Entity-resolution links that reach a *second, physical* California address

These are the strongest cases: Senzing merged or linked records across premises, so the business demonstrably operates at more than one site.

| Business near the hotel | Distance from hotel | Linked to | Key |
|---|---|---|---|
| 2669 **LUXOR FLOORS INC**, 1101 Broadway | 813 m | 600084 LUXOR FLOORS INC, 1325 Howard Ave (1,936 m away) | +NAME |
| 2670 **LUXOR FLOORS INC**, 1717 Adrian Rd | 1,171 m | same 600084 record (3,683 m from that site) | +NAME |
| 2484 **KS CHANNING ROAD ELECTRIC**, 1291 Whitehorn Way | 486 m | 601891, 18 Channing Road (2,874 m) | +NAME |
| 2382 **KAMRAN EHSANIPOUR AIA** (1633 Bayshore) | 732 m | 601593 EHSANIPOUR KAMRAN, 2855 Adeline Dr (3,613 m) | +NAME |
| 36 **836-868 COWAN RD LLC**, 868 Cowan Rd | 1,383 m | 35, 836 Cowan Rd (3,328 m) | +NAME |

Luxor Floors is the clearest: two separate shopfront entities (813 m and 1,171 m from the hotel) both tie by **+NAME** to one state contractor record — one firm, multiple premises.

### 2. Links to state-level records with no street address (mail-drop or licence-only)

These say "the local shop is the local face of a licensed statewide entity," though they are **+NAME** only, so the match survives a move but gives no second location:

- 2869 **MID PENINSULA ROOFING AND SOLAR**, 1239 Rollins Rd (461 m) → 600469 PENINSULA ROOFING, P O BOX 141
- 3640 **RIVER ENGINEERING INC**, 1300 Rollins Rd (355 m) → 600191, P O BOX 1367
- 2888 **MILLER DEVELOPMENT**, 809 California Dr (1,335 m) → 603227 MILLER OTTO DEVELOPMENT INC, P O BOX 121
- 2867 **MID PENINSULA MANAGEMENT INC** → 602358, P O BOX 1652
- 3011 **NESCO HEATING AND AIR CONDITIONING** → 602714, 1534 Plaza Lane

And a long tail with **no address on file at all**, all +NAME: 1561 ENCORE GROUP USA LLC (in the hotel itself, 0 m) → 600708; 124 ACE PARKING MANAGEMENT INC (765 Airport Blvd, 1,381 m) → 602552; 3163 ORKIN SERVICES OF CALIFORNIA INC (1815 Bayshore, 1,253 m) → 601034; 773 BRIGHTVIEW LANDSCAPE SERVICES (825 Mahler, 394 m) → 600863; 1694 FIREMASTER (825 Mahler, 394 m) → 603160; 895 CAL STEAM INC (1003 Edwards Rd, 495 m) → 602026 "C S D M INC DBA CAL STEAM"; 2404 KDDI AMERICA INC → 602717; 1134 COIT SERVICES INC (1588 Gilbreth, 602 m) → 602032 C & S DRAPERIES INC; plus 81 AA CUSTOMS PROAV, 463 AUDIO IMAGES→HENRYS AUDIO VISUAL, 564 BAY AREA SERVICE→BAY AREA HVAC & REFRIGERATION, 901 CALICO PLUMBING, 1331 DELAHOUSSAYE CONSTRUCTION, 1510 EK ROBIN CONSTRUCTION, 3272 PAUL COFFEY CONSTRUCTION, 3934 SIGNATURE CONSTRUCTION, 2094 HOME → MILLENNIUM DWELLINGS / HOME COMPANY INC.

### 3. Multi-site operators *inside* this neighbourhood

- 308 and 309 **AMERICAN MEDICAL RESPONSE WEST** — 1616 Rollins Rd (920 m) and 1510 Rollins Rd (485 m), linked **+NAME**: one ambulance operator on two yards.
- 2832 and 2837 **META PLATFORMS INC**, both 1832 Rollins Rd (1,500 m), linked +NAME — two records, same building.
- 3852 / 3853 **SEGAN INDUSTRIES INC**, 810 Stanton Rd (862 m), +NAME.
- 1254 **CUISINE STYLE** (BURLINGAME) ↔ 100061 **CUISINE STYLE** (ABC-RETAIL), both 810 Stanton Rd, +NAME — the caterer's liquor licence resolved to the same business.
- 1391 **DIVINEY CONSTRUCTION INC** (802 Burlway Rd, 263 m) ↔ 603131, +NAME; 427 **ART OF ITALY INC** (1003 Edwards Rd, 495 m) ↔ 602815, +NAME; 1825 **GENERATION BUILDERS INC** ↔ 601370 GB PLUMBING & CONSTRUCTION LLC by **+ADDRESS+PHONE**; 2723 MARCHETTI GROUP ↔ 603225 MORRONE CONSTRUCTION, also +ADDRESS+PHONE (both at 890 Cowan Rd, 1,333 m).

### A caveat on the obvious brands

Plenty of names in the cells are clearly outposts of national or statewide firms — 2202 IRON MOUNTAIN and 1993 HALLMARK MARKETING CORPORATION in the hotel building itself (0 m), 4394 TRAVEL TRADERS #698 (0 m), 1260 CUSHMAN & WAKEFIELD U S INC (136 m), 4246 TESLA INC (272 m), 1641 EXTRA SPACE STORAGE #1461 (461 m), 1041 CHEVRON STATIONS INC #1504 (813 m), 30 7-ELEVEN (1,050 m), 3236 PANASONIC AVIONICS (1,381 m), 4718 XEROX CORP (1,163 m), 3469 PUBLIC STORAGE INC (1,171 m). But for these the context carries **no entity-resolution link** to any other site — the inference rests only on the name as printed in the Burlingame business-licence record, not on evidence in this graph.

In [24]:
ask("Is anything in this data wrong or suspicious?")

Question: Is anything in this data wrong or suspicious?
Searching within 5000 m (no distance given, so searching every geocoded business)
Context: 62,690 chars (~15,672 tokens)


Yes — several things. Ranked by how much they should change your reading of the map.

**1. Five entities carry a second, un-geocodable address (flagged in the data-quality block).**
Each has a Placekey whose "what" starts with `0`, and in every case the "where" falls into the *same* cell `5vg-7y7-f9f` — that is a fallback/no-match bucket, not a real location. Treat those coordinates as meaningless:
- 653 BENIHANA OF TOKYO INC (1496 vs 1494 BAYSHORE HWY)
- 1359 DEVINCENZI ARCHITECTURAL PRODUCTS (1717 ADRIAN RD vs "1717 ADVIAN ROAD" — typo)
- 1918 GRAND HARBOR SEAFOOD AND DIMSUM
- 2076 HILLS POOL SERVICE INC (23 EDWARDS CT vs "23 EDWRADS CT" — typo)
- 3182 P L ANNUZZI INC (827 MALCOLM RD vs "827 MALCOM ROAD" — typo)

The typos are the likely cause of the match failure; the underlying businesses are probably fine.

**2. One of those conflicts actually moves a restaurant.** Entity 1918 GRAND HARBOR has "1492 OLD BAYSHORE HWY" → cell `5vg-7y3-xqz` (409 m) *and* "1492 BAYSHORE HWY" → cell `5vg-7y4-t9z`, which is the 778 BURLWAY RD cell at 355 m. Both are real cells, so the 409 m figure is not safe to quote; the restaurant is somewhere in the 355–410 m band. Note also 653 BENIHANA's good record sits at 1496 BAYSHORE HWY, 486 m — so two near-identical street numbers resolve to three different cells in this corner.

**3. A self-contradictory distance.** 2669 LUXOR FLOORS INC and 2670 LUXOR FLOORS INC both link +NAME to the same external entity 600084 at 1325 HOWARD AVENUE 337 — but that one address is reported as 1936 m away from one and 3683 m from the other. The same address cannot be at two distances from one starting point. At least one of those is wrong, and 2669 (1101 BROADWAY, 813 m) / 2670 (1717 ADRIAN RD, 1171 m) look like the same firm recorded twice.

**4. Clear under-merging: the same business appearing as two entities with no merge.**
- 3880 SF AIRPORT MARRIOTT (BURLINGAME) and 100137 MARRIOTT HOTEL (ABC-RETAIL) share cell `5vg-7y4-575` at 950 m — almost certainly one hotel, two entities. Compare 1415 DOUBLETREE, where the city and ABC records *were* merged (3 records). The merging rule is being applied inconsistently.
- Same pattern at 1177 AIRPORT BLVD (656 m): 100050 CROWNE PLAZA (ABC-RETAIL) vs 1855 GLOBAL CITY HOTELS INC / 2861 MICHAELS HAIR STYLING-CROWNE PLAZA (BURLINGAME), unmerged.
- 372 APEX MARITIME CO INC (825 Mahler, 394 m) and 373 APEX MARITIME CO INC (1534 Rollins, 709 m) — same name, no ER link listed.
- 3452 PROLINE AUTO BODY (1300 Rollins), 3451 PROLINE AUTO BODY (1291 Whitehorn), 3453 PROLINE AUTO COLLISION CENTER (1313 N Carolan) — three entities, no links.
- 2832/2837 META PLATFORMS and 3852/3853 SEGAN INDUSTRIES are +NAME-linked but still separate entities at the *same* address — duplicates that were recognised and not collapsed. Six PALO ALTO MEDICAL FOUNDATION entities (3225, 3227–3231) at 1720 EL CAMINO REAL likewise.
- 308 and 309 AMERICAN MEDICAL RESPONSE WEST are +NAME-linked but sit in different cells (1616 ROLLINS, 920 m; 1510 ROLLINS, 485 m) — plausible as two depots, but worth doubting.

**5. Weak links that shouldn't be read as business relationships.** Everything hanging off 9 1799 BUILDING-GELLER & PREINER (to 33, 322, 464, 1820, 3015, 3448, 3929, 4709) is **+ADDRESS only** — that is just "same office building, 1799 Bayshore", 1122 m away. Same for 8 1501 ADRIAN RD LLC ↔ 4269 THE FLOOR STORE. These are landlord/tenant co-location, not shared ownership.

**6. Links worth a second look because the names differ but the phone matches:** 2723 MARCHETTI GROUP ↔ 603225 MORRONE CONSTRUCTION (+ADDRESS+PHONE, 890 Cowan Rd) and 1825 GENERATION BUILDERS ↔ 601370 GB PLUMBING (+ADDRESS+PHONE). A shared phone *and* address usually means one operator trading under two names — but it can also mean a stale answering service or a shared suite, so it is a lead, not a fact.

**7. Senzing itself says it is unsure** on 1044 CHIC ↔ 1045/1046, and 2559 LE REVE SALON ↔ 2560/2561/2562 — all AMBIGUOUS, all +NAME+ADDRESS. These look like individual stylists renting chairs, recorded as separate licences; the 4/5/1604/4154 cluster at 1200 CAPUCHINO is the same phenomenon but linked by +ADDRESS alone.

**8. Two different hotel brands in one cell.** 1415 DOUBLETREE HOTEL SAN FRAN AIRPORT and 100124 COURTYARD BY MARRIOTT SAN FRANCISCO AIRPORT - BURLINGAME both sit at 765 AIRPORT BLVD, cell `5vg-7y5-2hq`, 1381 m. Sharing a cell means sharing a building; two full-service airport hotels in one building is unlikely, so one of the two addresses is probably mis-recorded.

**9. Minor oddity at the starting point.** The Hyatt's own cell (`5vg-7y4-wx5`, 0 m) contains 1993 HALLMARK MARKETING CORPORATION and 2202 IRON MOUNTAIN alongside the obvious hotel-services tenants (1259 CURTIN CONVENTION & EXPOSITION, 1561 ENCORE GROUP, 4394 TRAVEL TRADERS #698). A records-storage firm and a greeting-card corporation holding a Burlingame licence at a hotel address is plausible (gift shop, service contract) but not verifiable from this context.

**Caveat on completeness:** many cells are truncated ("… and 20 more"), so absence of a business from this list is not evidence it isn't there.

In [27]:
ask("What could you tell me here that a keyword search over the raw "
    "licence records could not?")

Question: What could you tell me here that a keyword search over the raw licence records could not?
Searching within 5000 m (no distance given, so searching every geocoded business)
Context: 62,690 chars (~15,672 tokens)


Here are the things this graph knows that a text search across the raw ABC / city / contractor files would miss.

**1. Who is physically inside the hotel, not just who shares a street name**
Five businesses sit in the *same Placekey cell* as the Hyatt Regency (entity 2120, cell 5vg-7y4-wx5, 0 m): Curtin Convention & Exposition Service (1259), Encore Group USA (1561), Hallmark Marketing (1993), Iron Mountain (2202) and Travel Traders #698 (4394) — i.e. the event/AV contractors and the gift-shop operator working out of the building. Keyword searching "1333 Bayshore" might find them; it would not tell you that 1496 Bayshore Hwy (Benihana, 653) is 486 m away while 1250 Bayshore (Max's, 2783) is 263 m — street numbers do not scale to metres.

**2. Real walking order, computed from coordinates**
Nearest licensed food/drink: Max's of Burlingame (2783, ABC-RETAIL, 263 m, in the Holiday Inn Express building 2090), Grand Harbor Seafood (1918, 409 m), Benihana (653, 486 m), H L Peninsula Pearl (1975, 577 m), H L Peninsula Hot Pot (1974, 692 m), Maverick Jacks (2781, 692 m), Weimax Wines & Spirits (4631, 711 m), then the Broadway strip at 1150 Paloma (973 m) with Behans Irish Pub (642), Bistro Arancini (694), Broadway Prime Rib (788), Cafe Figaro (882). Topgolf (4372) is 1,152 m. That ranking is not recoverable from text.

**3. One business, several records — and several businesses, one name**
The Hyatt itself is three source records merged (BURLINGAME:20500020 plus ABC-RETAIL 220571-47 and -58): two licence hits, one premises. Conversely, American Medical Response West appears as entity 308 (1616 Rollins, 920 m) and 309 (1510 Rollins, 485 m), linked **+NAME** — same operator, two depots, which a keyword hit list would show as duplicates with no verdict. Meanwhile three similar names — Proline Auto Body 3452 (1300 Rollins, 355 m), Proline Auto Body 3451 (1291 Whitehorn, 486 m) and Proline Auto Collision Center 3453 (1313 N Carolan, 546 m) — were *not* merged; the graph is telling you not to treat them as one record.

**4. Links that survive a name change or an address change**
- Cuisine Style: city record 1254 at 810 Stanton Rd (862 m) is tied **+NAME** to ABC-RETAIL entity 100061 — that is how you know the licence belongs to that caterer at that building.
- 2723 MARCHETTI GROUP ↔ 603225 MORRONE CONSTRUCTION, **+ADDRESS+PHONE** (890 Cowan Rd, 1,333 m) — same phone, different trading name.
- 600206 OTOOLE CONSTRUCTION ↔ 602668 D O T CONSTRUCTION ↔ 603089 DOT CONCRETE, all **+ADDRESS+PHONE** at 1657 Balboa Way.
- 1825 GENERATION BUILDERS ↔ 601370 GB PLUMBING & CONSTRUCTION, **+ADDRESS+PHONE**.
No keyword query joins "O'Toole" to "Dot Concrete".

**5. Explicit uncertainty**
Senzing marks the salon chair-renters AMBIGUOUS: 2559 LE REVE SALON vs 2560/2561/2562 (+NAME+ADDRESS), and 1044 CHIC vs 1045/1046. At 1200 Capuchino the 1202 Hair Studio entries (4, 5, 1604, 4154) are joined by **+ADDRESS only** — shared suite, not proven shared business. A flat search would silently merge or silently split these.

**6. Coordinates you should distrust**
Placekeys whose "what" starts with 0 failed address matching:
- Benihana (653) has 1496 Bayshore Hwy → 222@5vg-7y4-syv *and* 1494 Bayshore Hwy → 05iualc3ju@…; the 486 m figure rests on the good record.
- Grand Harbor Seafood (1918) is worse: its 409 m placement comes from an unreliable 0-prefixed key (1492 OLD BAYSHORE HWY), while its second record maps to cell 5vg-7y4-t9z — the 355 m cell. Treat its distance as roughly 350–410 m, not exact.
- Same pattern for 1359 Devincenzi, 2076 Hills Pool Service, 3182 P L Annuzzi.

**7. What the sources do *not* license**
Alcohol is implied only by ABC-RETAIL / ABC-NONRETAIL. So Burlingame Liquors (839, 1,025 m), Adeline Market (147) and 7-Eleven (30, 1,050 m) appear here **without** an ABC record merged in — I cannot tell you they sell alcohol from this data, even though the names suggest it. Equally, ABC-NONRETAIL entries like International Beverage (200006, 709 m) and Legacy Through Giving Foundation (200005, 478 m) are not places to drink.

**8. Competitor/overflow hotel set as entities, not strings**
Holiday Inn Express (2090) 263 m; Bay Landing (585) 577 m; Crowne Plaza (100050) 656 m — same cell as its operator Global City Hotels (1855) and the in-house salon Michaels (2861); Vagabond Inn (4509) 699 m; SF Airport Marriott — city entity 3880 and ABC entity 100137 share cell 5vg-7y4-mzf… (1699 Bayshore, 950 m) but were *not* merged, with King Street Cafe (100157) in the same building; Hampton Inn (2000) 1,077 m; DoubleTree (1415) and Courtyard (100124) sharing 765 Airport Blvd at 1,381 m; Embassy Suites/Djont (100054) 1,494 m; Westin (100113) 1,444 m; Aloft (100115) 1,514 m.

In [ ]:
ask("I want to get a drink around here.  What are my options within 1000m")

In [ ]:
ask("Find a bar not in a hotel that is within 1000 m")

In [ ]:
ask("Tell me about Top Golf and can I get a drink there?")